# Ranking and webhook checks, reviewed

The original notebook's webhook attempts both returned 403. They do not establish a verification bypass. Its signature omitted the message ID and timestamp, used the encoded secret as the key, sent a hexadecimal digest under the wrong header, and serialized the sent body differently from the signed body. See [Svix verification](https://docs.svix.com/receiving/verifying-payloads/manual).

The original ranking "success rate" counted any HTTP response, including 4xx and 5xx. "Slow Burn" accumulated coroutines before starting them, and "Marathon" was another burst. The limit parameter is ignored by the ranking route. Recorded wall times include connection contention; they do not isolate database or server time. The final 0% result discarded errors and cannot identify the cause.

This replacement defaults to localhost, makes at most 100 requests with at most 10 workers, and reports status codes and transport errors. Keep the original attachment as historical evidence. Python 3.11+; no third-party packages required.


In [ ]:
import os
BASE_URL = os.environ.get("RANKING_TEST_BASE_URL", "http://localhost:3000")
RUN_RANKING_TESTS = os.environ.get("RUN_RANKING_TESTS") == "1"
RUN_WEBHOOK_CHECKS = os.environ.get("RUN_WEBHOOK_CHECKS") == "1"


In [ ]:
"""Bounded ranking measurements and exact-byte Svix signing. Standard library only."""
import asyncio
import base64
from collections import Counter
import hashlib
import hmac
import json
import math
import statistics
import time
import urllib.error
import urllib.request
import uuid

ENDPOINTS = (
    "/api/v1/challenges/black-box/ranking",
    "/api/v1/challenges/broken-agent/ranking",
)


def signed_webhook(payload, secret, message_id=None, timestamp=None):
    if not secret.startswith("whsec_"):
        raise ValueError("Expected a whsec_ signing secret")
    key = base64.b64decode(secret[6:], validate=True)
    if not key:
        raise ValueError("Signing secret must not be empty")
    body = json.dumps(payload, separators=(",", ":"), ensure_ascii=False).encode()
    message_id = message_id or "msg_" + uuid.uuid4().hex
    if timestamp is None:
        timestamp = int(time.time())
    timestamp = str(timestamp)
    message = f"{message_id}.{timestamp}.".encode() + body
    signature = base64.b64encode(hmac.new(key, message, hashlib.sha256).digest()).decode()
    return body, {
        "Content-Type": "application/json",
        "svix-id": message_id,
        "svix-timestamp": timestamp,
        "svix-signature": "v1," + signature,
    }


def request_once(url, *, body=None, headers=None, timeout=10):
    started = time.perf_counter()
    status = None
    error = None
    content = b""
    try:
        request = urllib.request.Request(url, data=body, headers=headers or {})
        with urllib.request.urlopen(request, timeout=timeout) as response:
            status = response.status
            content = response.read()
    except urllib.error.HTTPError as exception:
        status = exception.code
        content = exception.read()
    except (urllib.error.URLError, TimeoutError, OSError) as exception:
        error = f"{type(exception).__name__}: {exception}"
    return {
        "url": url,
        "status": status,
        "error": error,
        "latency_ms": (time.perf_counter() - started) * 1000,
        "body_size": len(content),
    }


def summarize(results):
    statuses = Counter(str(row["status"]) for row in results if row["status"] is not None)
    errors = Counter(row["error"] for row in results if row["error"] is not None)
    successes = sum(row["status"] is not None and 200 <= row["status"] < 300 for row in results)
    latencies = sorted(row["latency_ms"] for row in results)
    return {
        "requests": len(results),
        "successes_2xx": successes,
        "success_rate_percent": 100 * successes / len(results) if results else 0,
        "http_statuses": dict(statuses),
        "transport_errors": dict(errors),
        "mean_latency_ms": statistics.mean(latencies) if latencies else None,
        "p50_latency_ms": latencies[math.ceil(len(latencies) * 0.50) - 1] if latencies else None,
        "p95_latency_ms": latencies[math.ceil(len(latencies) * 0.95) - 1] if latencies else None,
        "max_latency_ms": max(latencies) if latencies else None,
    }


async def run_experiment(base_url="http://localhost:3000", *, total_requests=20, concurrency=4, interval=0):
    if not isinstance(total_requests, int) or not 1 <= total_requests <= 100:
        raise ValueError("total_requests must be an integer between 1 and 100")
    if not isinstance(concurrency, int) or not 1 <= concurrency <= 10:
        raise ValueError("concurrency must be an integer between 1 and 10")
    if not math.isfinite(interval) or interval < 0:
        raise ValueError("interval must be finite and nonnegative")
    queue = asyncio.Queue(maxsize=concurrency)
    results = []

    async def worker():
        while True:
            item = await queue.get()
            try:
                if item is None:
                    return
                request_id, url, queued_at = item
                dispatched_at = time.perf_counter()
                result = await asyncio.to_thread(request_once, url)
                result["request_id"] = request_id
                result["queue_ms"] = (dispatched_at - queued_at) * 1000
                results.append(result)
            finally:
                queue.task_done()

    started = time.perf_counter()
    async with asyncio.TaskGroup() as group:
        for _ in range(concurrency):
            group.create_task(worker())
        for request_id in range(total_requests):
            if request_id and interval:
                await asyncio.sleep(interval)
            endpoint = ENDPOINTS[request_id % len(ENDPOINTS)]
            await queue.put((request_id, base_url.rstrip("/") + endpoint, time.perf_counter()))
        for _ in range(concurrency):
            await queue.put(None)
        await queue.join()
    elapsed = time.perf_counter() - started
    summary = summarize(results)
    summary["duration_seconds"] = elapsed
    summary["requests_per_second"] = len(results) / elapsed
    summary["by_endpoint"] = {
        endpoint: summarize([row for row in results if row["url"].endswith(endpoint)])
        for endpoint in ENDPOINTS
    }
    return summary, sorted(results, key=lambda row: row["request_id"])


## Ranking measurements

Counts below are totals across both endpoints, not per endpoint. Workers start during production of requests, so the paced experiment sends traffic while waiting. Each request has a 10-second timeout. Latency includes the request and body read but excludes the separately reported queue wait. Percentiles include failed requests. The measurements do not establish the server's capacity.


In [ ]:
if RUN_RANKING_TESTS:
    for name, config in {
        "Small burst": {"total_requests": 20, "concurrency": 4},
        "Paced requests": {"total_requests": 20, "concurrency": 2, "interval": 0.25},
    }.items():
        summary, results = await run_experiment(BASE_URL, **config)
        print(name)
        print(json.dumps(summary, indent=2))
else:
    print("Ranking checks skipped. Set RUN_RANKING_TESTS=1 to run against BASE_URL.")


## Webhook signature checks

Read CLERK_WEBHOOK_SIGNING_SECRET from the environment; do not print it. Use the exact bytes returned by signed_webhook for the request. The synthetic session.ended event deliberately avoids this repository's session.created participant and email flow. A successful response tests signature verification and acknowledgement only, not the full session-created pipeline. A rejection from a proxy does not establish what the handler did.


In [ ]:
if RUN_WEBHOOK_CHECKS:
    secret = os.environ["CLERK_WEBHOOK_SIGNING_SECRET"]
    payload = {"type": "session.ended", "object": "event", "data": {}}
    body, valid_headers = signed_webhook(payload, secret)
    _, invalid_headers = signed_webhook(payload, "whsec_" + base64.b64encode(b"test-wrong-secret").decode())
    for name, headers in [("Invalid signature", invalid_headers), ("Valid signature", valid_headers)]:
        result = await asyncio.to_thread(request_once, BASE_URL.rstrip("/") + "/api/webhooks/clerk", body=body, headers=headers)
        print(name, json.dumps(result, indent=2))
else:
    print("Webhook checks skipped. Set RUN_WEBHOOK_CHECKS=1 with a test signing secret.")


## Server action and verification

The public ranking API now caches ranking data for five seconds after a calculation completes. Concurrent reads for the same challenge share one pending calculation. Unknown challenge slugs are rejected before allocating cache entries. Failures are not cached. Each HTTP response keeps its own versioned envelope and request ID with Cache-Control: no-store.

The cache applies only to the public ranking API. Admission, placement, admin reads, and existing internal ranking callers continue to query fresh data. The cache is per running server process, so multiple replicas each compute their own result. Public names, withdrawals, scores, and challenge status may take up to five seconds after a calculation completes to appear in this API. This reduces duplicate queries but does not prove the cause of the recorded production delays or supply a distributed rate limit.

Verified locally: 14 ranking tests, including 500 simultaneous callers, expiry, distinct challenges, unknown slugs, and retry after failure. Four notebook checks cover real local HTTP status handling, bounded concurrency, paced dispatch, transport failure reporting, notebook execution with network checks disabled, and Python signatures against the installed Clerk verifier. Web type checks and lint for changed TypeScript files passed.

No production load was generated during this review, and the code change has not been deployed. The original attachment is preserved.
